# 🍎 Fruit Image Classification Using CNN

**Objective:** Build a CNN that identifies fruits (Apple, Banana, Orange, Strawberry) from images.

**Input:** Fruit image  
**Output:** Predicted fruit name + confidence score

In [3]:
import os
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
os.environ['KAGGLE_KEY'] = getpass('Kaggle key: ')

Kaggle username: nusratanwar
Kaggle key: ··········


In [4]:
!kaggle datasets download -d moltean/fruits

Dataset URL: https://www.kaggle.com/datasets/moltean/fruits
License(s): CC-BY-SA-4.0
100% 7.96G/7.96G [01:08<00:00, 126MB/s]



In [5]:
!unzip fruits.zip -d fruits

Streaming output truncated to the last 5000 lines.
  inflating: fruits/fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 4/r0_221.jpg  
  inflating: fruits/fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 4/r0_225.jpg  
  inflating: fruits/fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 4/r0_229.jpg  
  inflating: fruits/fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 4/r0_233.jpg  
  inflating: fruits/fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 4/r0_237.jpg  
  inflating: fruits/fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 4/r0_241.jpg  
  inflating: fruits/fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 4/r0_245.jpg  
  inflating: fruits/fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 4/r0_249.jpg  
  inflating: fruits/fruits-360_original-size/fruits-360-original-size/Validation/Raspberry 4/r0_25.jp

In [6]:
!ls "fruits/fruits-360_100x100/fruits-360/Training"

'Almonds 1'		 'Cucumber 12'		  'Pear 14'
'Apple 10'		 'Cucumber 13'		  'Pear 2'
'Apple 11'		 'Cucumber 3'		  'Pear 3'
'Apple 12'		 'Cucumber 4'		  'Pear 5'
'Apple 13'		 'Cucumber 5'		  'Pear 6'
'Apple 14'		 'Cucumber 6'		  'Pear 7'
'Apple 17'		 'Cucumber 7'		  'Pear 8'
'Apple 18'		 'Cucumber 8'		  'Pear 9'
'Apple 19'		 'Cucumber 9'		  'Pear Abate 1'
'Apple 20'		 'Dates 1'		  'Pear Forelle 1'
'Apple 21'		 'Dates 2'		  'Pear Kaiser 1'
'Apple 22'		 'Eggplant 1'		  'Pear Monster 1'
'Apple 23'		 'Eggplant long 1'	  'Pear Red 1'
'Apple 5'		 'Fig 1'		  'Pear Stone 1'
'Apple 6'		 'garlic bulb 1'	  'Pear Williams 1'
'Apple 7'		 'Ginger 2'		  'Pepino 1'
'Apple 8'		 'Ginger Root 1'	  'Pepper 1'
'Apple 9'		 'Gooseberry 1'		  'Pepper 2'
'Apple Braeburn 1'	 'Granadilla 1'		  'Pepper Green 1'
'Apple Crimson Snow 1'	 'Grape 1'		  'Pepper Orange 1'
'Apple Golden 1'	 'Grape Blue 1'		  'Pepper Orange 2'
'Apple Golden 2'	 'Grapefruit Pink 1'	  'Pepper Red 1'
'Apple Golden 3'	 'Grapefruit White 1'	  'Peppe

In [7]:
!mkdir -p myfolder/sub

In [8]:
!mkdir -p dataset/train dataset/test

## Step 2: Load and Prepare Dataset

Selected 4 fruit classes (Apple, Banana, Orange, Strawberry) from Fruits-360.
Copied them into a clean `dataset/` folder with `train` and `test` splits.

In [9]:
BASE = "fruits/fruits-360_100x100/fruits-360"

!cp -r "$BASE/Training/Apple 10" "dataset/train/Apple"
!cp -r "$BASE/Training/Banana 1" "dataset/train/Banana"
!cp -r "$BASE/Training/Orange 1" "dataset/train/Orange"
!cp -r "$BASE/Training/Strawberry 1" "dataset/train/Strawberry"

!cp -r "$BASE/Test/Apple 10" "dataset/test/Apple"
!cp -r "$BASE/Test/Banana 1" "dataset/test/Banana"
!cp -r "$BASE/Test/Orange 1" "dataset/test/Orange"
!cp -r "$BASE/Test/Strawberry 1" "dataset/test/Strawberry"

In [10]:
!ls dataset/train

Apple  Banana  Orange  Strawberry


## Step 3: Preprocess and Load Data

Resize all images to a fixed size, scale pixel values to 0–1,
and load them in batches using Keras. Folder names become class labels.

In [11]:
import tensorflow as tf
print(tf.__version__)

2.20.0


In [12]:
train_ds =tf.keras.utils.image_dataset_from_directory(
    "dataset/train",
    image_size=(100,100),
    batch_size=32
)

Found 2160 files belonging to 4 classes.


In [13]:
test_ds =tf.keras.utils.image_dataset_from_directory(
    "dataset/test",
    image_size=(100,100),
    batch_size=32
    )


Found 721 files belonging to 4 classes.


In [14]:
class_names = train_ds.class_names
print(class_names)

['Apple', 'Banana', 'Orange', 'Strawberry']


### STEP 4: BUILD THE CNN MODEL
stack layers using keras sequential: Rescaling (normalize pixels),
cnv2D + maxpooling (extract features), Flatten, then Dense layers
for the final 4- class prediction

In [15]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(100,100,3)),
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Conv2D(16,3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(4, activation='softmax')
])

##STEP 5: comole the model
configure the model for training: choose an optimizer (adam),
a loss function for multi-class labels, and track accuracy

In [16]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

##STEP 6: TRAIN THE MODEL
train the CNN on the training data for a few  epochs,
and check accuracy on the test data after each epoch.

In [17]:
model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10
)

Epoch 1/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 11s 143ms/step - accuracy: 0.8958 - loss: 0.4052 - val_accuracy: 1.0000 - val_loss: 0.0136
Epoch 2/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 11s 155ms/step - accuracy: 1.0000 - loss: 0.0022 - val_accuracy: 1.0000 - val_loss: 0.0057
Epoch 3/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 21s 159ms/step - accuracy: 1.0000 - loss: 8.8286e-04 - val_accuracy: 1.0000 - val_loss: 0.0020
Epoch 4/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 19s 139ms/step - accuracy: 1.0000 - loss: 2.4226e-04 - val_accuracy: 1.0000 - val_loss: 0.0010
Epoch 5/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 11s 160ms/step - accuracy: 1.0000 - loss: 8.2115e-05 - val_accuracy: 1.0000 - val_loss: 8.4254e-04
Epoch 6/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 11s 163ms/step - accuracy: 1.0000 - loss: 3.4629e-05 - val_accuracy: 1.0000 - val_loss: 7.3740e-04
Epoch 7/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 21s 169ms/step - accuracy: 1.0000 - loss: 2.0523e-05 - val_accuracy: 1.0000 - val_loss: 4.7343e-04
Epoch 8/10
68/68 ━━━━━━━━━━━━━━━━━━━━ 10s 143ms/step - accuracy: 

## Step 7: Predict on a New Image

Load a single fruit image, run it through the trained model,
and print the predicted fruit name with its confidence score.

In [18]:
img = tf.keras.utils.load_img("dataset/test/Apple/r0_103_100.jpg", target_size=(100, 100))
arr = tf.keras.utils.img_to_array(img)
arr = tf.expand_dims(arr,0)

In [19]:
!ls dataset/test/Apple | head -1

r0_103_100.jpg


In [20]:
preds = model.predict(arr)
index = tf.argmax(preds[0])
confidence = tf.reduce_max(preds[0])
print(class_names[index], float(confidence))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
Apple 0.9999994039535522


## Step 8: Classification Report

Collect the true labels and the model's predictions on the test data,
then use scikit-learn to show precision, recall, and F1-score per fruit.

In [21]:
import numpy as np
from sklearn.metrics import classification_report
y_true = []
y_pred =[]
for images, labels in test_ds:
  preds = model.predict(images)
  y_true.extend(labels.numpy())
  y_pred.extend(np.argmax(preds, axis=1))
print(classification_report(y_true, y_pred, target_names=class_names))



1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 623ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 223ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
              precision    recall  f1-score   support

       Apple       1.00      1.00      1.00       231
    